In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Load Clean Train Data
# KODE BARU YANG BENAR:
df_train = pd.read_csv('../data/processed/volve_train_clean.csv', sep=';', decimal=',')

target_col = 'AVG_CHOKE_SIZE_P'
feature_cols = [
    'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 
    'AVG_DOWNHOLE_PRESSURE', 'AVG_WELLHEAD_PRESSURE', 'AVG_TEMPERATURE'
]

# JURUS SAPU JAGAT ANTI-NaN
# Memastikan tidak ada data sensor yang kosong terlewat di data training
df_train[feature_cols] = df_train[feature_cols].bfill().ffill()

X = df_train[feature_cols]
y = df_train[target_col]

# 2. Data Splitting (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. NORMALISASI DATA (Min-Max Scaler)
scaler = MinMaxScaler()
# Fit hanya pada X_train untuk mencegah Data Leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Pembangunan Model (Menggunakan Data yang Sudah Dinormalisasi)
# Model 1: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)

# Model 2: Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_preds = rf_model.predict(X_test_scaled)

# Model 3: XGBoost
xgb_model = XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train_scaled, y_train)
xgb_preds = xgb_model.predict(X_test_scaled)

# 5. Evaluasi Metrik Performa Model
def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"--- Performa {name} ---")
    print(f"RMSE : {rmse:.4f}\nMAE  : {mae:.4f}\nR2   : {r2:.4f}\n")

evaluate_model("Linear Regression", y_test, lr_preds)
evaluate_model("Random Forest", y_test, rf_preds)
evaluate_model("XGBoost", y_test, xgb_preds)

# 6. Analisis Feature Importance (Kita pakai Random Forest)
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices] * 100, y=np.array(feature_cols)[indices], palette="viridis")
plt.title("Feature Importance: Penentu Bukaan Choke (Random Forest)", fontweight='bold')
plt.xlabel("Tingkat Kepentingan (%)")
plt.show()

# 7. SIMPAN MODEL TERBAIK DAN SCALER (Krusial!)
# Asumsi Random Forest menang, kita simpan RF dan Scaler-nya
joblib.dump(rf_model, '../models/best_rf_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl') 
print("✅ Model terbaik dan Scaler berhasil disimpan di folder '../models/'")